# 03 - Analyses

**Objectif** : Répondre aux questions de recherche par des calculs numériques.  
Les graphiques sont produits dans `04_visualisations.ipynb`.

**Problématique** : Dans quelle mesure le niveau de diplôme influence-t-il l'accès à l'emploi ?

**Sous-questions** :
- SQ1 : Le diplôme protège-t-il de la même façon selon la région ?
- SQ2 : Les diplômes universitaires ont-ils tous la même valeur ?
- SQ3 : Tous les diplômes protègent-ils du chômage de la même manière ?

In [3]:
import pandas as pd
import numpy as np
from pathlib import Path

# Chemin du projet et des données
PROJECT_ROOT = Path.cwd().parent
CLEAN_DIR = PROJECT_ROOT / 'data' / 'clean'
RAW_DIR = PROJECT_ROOT / 'data' / 'raw'

# Chargement des données nettoyées
emploi  = pd.read_csv(CLEAN_DIR / 'emploi_clean.csv')
chomage = pd.read_csv(CLEAN_DIR / 'chomage_clean.csv')

print('Données bien chargées.')

Données bien chargées.


## 1. Fusion des deux datasets

On fusionne emploi et chômage sur `country`, `year` et `education_level` (inner join).

In [4]:
# Fusion sur les 3 clés communes
df = pd.merge(
    emploi [['country', 'year', 'education_level', 'nb_employes']],
    chomage[['country', 'year', 'education_level', 'nb_chomeurs']],
    on=['country', 'year', 'education_level'],
    how='inner'
)

# Calcul du taux de chômage
# Formule : taux = nb_chomeurs / (nb_employes + nb_chomeurs) * 100
# (la force de travail = employés + chômeurs)
df['taux_chomage'] = (
    df['nb_chomeurs'] / (df['nb_employes'] + df['nb_chomeurs'])
) * 100

print('Dataset fusionné :')
print(f'  Lignes  : {len(df):,}')
print(f'  Pays    : {df["country"].nunique()}')
print(f'  Période : {df["year"].min()} – {df["year"].max()}')
print()
display(df.head(9))

Dataset fusionné :
  Lignes  : 5,924
  Pays    : 175
  Période : 2000 – 2023



,country,year,education_level,nb_employes,nb_chomeurs,taux_chomage
0,AFG,2021,LOW,1273.143,81.747,6.033479
1,AFG,2021,MEDIUM,940.560,98.258,9.458635
2,AFG,2021,HIGH,448.800,48.743,9.796741
3,AFG,2020,LOW,910.916,124.640,12.036046
4,AFG,2020,MEDIUM,693.897,96.029,12.156708
5,AFG,2020,HIGH,386.350,65.025,14.405982
6,AFG,2017,LOW,1323.106,185.280,12.283328
7,AFG,2017,MEDIUM,748.874,144.286,16.154552
8,AFG,2017,HIGH,480.561,87.884,15.460423


## 2. Vue globale : volumes emploi et chômage par niveau

In [5]:
# Totaux cumulés (en milliards = milliers / 1 000 000)
totaux = df.groupby('education_level')[['nb_employes','nb_chomeurs']].sum().reindex(['LOW','MEDIUM','HIGH'])
totaux_md = totaux / 1e6

print('Volume cumulé mondial 2000–2023 (en milliards de personnes) :')
display(totaux_md.round(2))

Volume cumulé mondial 2000–2023 (en milliards de personnes) :


,nb_employes,nb_chomeurs
education_level,,
LOW,9.68,0.70
MEDIUM,9.32,0.85
HIGH,6.57,0.41


## 3. Analyse principale : Taux de chômage par niveau (SQ3)

In [6]:
# Taux de chômage moyen par niveau (sur tous les pays et toutes les années)
taux_par_niveau = (
    df.groupby('education_level')['taux_chomage']
    .mean()
    .reindex(['LOW','MEDIUM','HIGH'])
    .round(2)
)

print('Taux de chômage moyen par niveau (2000–2023) :')
for niveau, taux in taux_par_niveau.items():
    print(f'  {niveau:8s}: {taux:.2f}%')

print()
ecart = taux_par_niveau['LOW'] - taux_par_niveau['HIGH']
print(f'Écart LOW→HIGH : {ecart:.2f} points de pourcentage')
print()
print('→ RÉPONSE SQ3 : Non, le taux de chômage diminue significativement avec le niveau de diplôme.')

Taux de chômage moyen par niveau (2000–2023) :
  LOW     : 10.92%
  MEDIUM  : 9.44%
  HIGH    : 6.90%

Écart LOW→HIGH : 4.02 points de pourcentage

→ RÉPONSE SQ3 : Non, le taux de chômage diminue significativement avec le niveau de diplôme.


## 4. Évolution temporelle du taux de chômage (2000–2023)

In [7]:
# Taux moyen par année et par niveau
taux_temporel = (
    df.groupby(['year', 'education_level'])['taux_chomage']
    .mean()
    .reset_index()
    .round(2)
)

# Vérification : impact de la crise 2008 sur chaque niveau
print('Taux moyen avant et après la crise de 2008 :')
avant = df[df['year'] < 2008].groupby('education_level')['taux_chomage'].mean().reindex(['LOW','MEDIUM','HIGH'])
apres = df[(df['year'] >= 2008) & (df['year'] <= 2012)].groupby('education_level')['taux_chomage'].mean().reindex(['LOW','MEDIUM','HIGH'])

comparaison = pd.DataFrame({'avant 2008': avant, '2008–2012': apres})
comparaison['variation'] = (comparaison['2008–2012'] - comparaison['avant 2008']).round(2)
display(comparaison.round(2))
print()
print('→ Tous les niveaux sont affectés par la crise, mais le niveau HIGH reste le plus résilient.')

Taux moyen avant et après la crise de 2008 :


,avant 2008,2008–2012,variation
education_level,,,
LOW,11.35,11.70,0.35
MEDIUM,9.34,10.06,0.72
HIGH,5.61,7.17,1.56



→ Tous les niveaux sont affectés par la crise, mais le niveau HIGH reste le plus résilient.


## 5. Analyse par région (SQ1)

In [8]:
# Classification des pays par région
REGIONS = {
    # Europe
    'FRA':'Europe','DEU':'Europe','ITA':'Europe','ESP':'Europe','GBR':'Europe',
    'POL':'Europe','NLD':'Europe','BEL':'Europe','SWE':'Europe','NOR':'Europe',
    'CHE':'Europe','AUT':'Europe','PRT':'Europe','GRC':'Europe','CZE':'Europe',
    'HUN':'Europe','ROU':'Europe','BGR':'Europe','HRV':'Europe','SVK':'Europe',
    'FIN':'Europe','DNK':'Europe','IRL':'Europe','LTU':'Europe','LVA':'Europe',
    'EST':'Europe','SVN':'Europe','RUS':'Europe','UKR':'Europe','SRB':'Europe',
    'ALB':'Europe','MDA':'Europe','MKD':'Europe','MNE':'Europe','BIH':'Europe',
    'ISL':'Europe','LUX':'Europe','MLT':'Europe','CYP':'Europe','BLR':'Europe',
    # Amériques
    'USA':'Amériques','CAN':'Amériques','MEX':'Amériques','BRA':'Amériques',
    'ARG':'Amériques','COL':'Amériques','CHL':'Amériques','PER':'Amériques',
    'ECU':'Amériques','BOL':'Amériques','PRY':'Amériques','URY':'Amériques',
    'PAN':'Amériques','CRI':'Amériques','GTM':'Amériques','DOM':'Amériques',
    'VEN':'Amériques','SLV':'Amériques','HND':'Amériques','JAM':'Amériques',
    # Asie-Pacifique
    'CHN':'Asie-Pacifique','JPN':'Asie-Pacifique','KOR':'Asie-Pacifique',
    'IND':'Asie-Pacifique','IDN':'Asie-Pacifique','THA':'Asie-Pacifique',
    'VNM':'Asie-Pacifique','PHL':'Asie-Pacifique','MYS':'Asie-Pacifique',
    'AUS':'Asie-Pacifique','NZL':'Asie-Pacifique','SGP':'Asie-Pacifique',
    'BGD':'Asie-Pacifique','PAK':'Asie-Pacifique','LKA':'Asie-Pacifique',
    'KHM':'Asie-Pacifique','MNG':'Asie-Pacifique','NPL':'Asie-Pacifique',
    # Afrique
    'ZAF':'Afrique','NGA':'Afrique','ETH':'Afrique','EGY':'Afrique',
    'DZA':'Afrique','MAR':'Afrique','TUN':'Afrique','GHA':'Afrique',
    'KEN':'Afrique','TZA':'Afrique','UGA':'Afrique','CMR':'Afrique',
    'CIV':'Afrique','SEN':'Afrique','ZMB':'Afrique','BWA':'Afrique',
    # Moyen-Orient
    'SAU':'Moyen-Orient','IRN':'Moyen-Orient','TUR':'Moyen-Orient',
    'IRQ':'Moyen-Orient','ISR':'Moyen-Orient','JOR':'Moyen-Orient',
    'KWT':'Moyen-Orient','ARE':'Moyen-Orient','QAT':'Moyen-Orient',
    'LBN':'Moyen-Orient','OMN':'Moyen-Orient','PSE':'Moyen-Orient',
}

df['region'] = df['country'].map(REGIONS)
df_region = df[df['region'].notna()].copy()

print(f'Pays avec région assignée : {df_region["country"].nunique()}')
print()
print('Pays par région :')
print(df_region.groupby('region')['country'].nunique().sort_values(ascending=False))

Pays avec région assignée : 99

Pays par région :
region
Europe            40
Amériques         20
Asie-Pacifique    16
Afrique           15
Moyen-Orient       8
Name: country, dtype: int64


In [9]:
# Taux de chômage moyen par région et par niveau
taux_region = (
    df_region.groupby(['region', 'education_level'])['taux_chomage']
    .mean()
    .reset_index()
    .round(2)
)

# Tableau croisé pour lecture facile
tableau_regions = taux_region.pivot(index='region', columns='education_level', values='taux_chomage')[['LOW','MEDIUM','HIGH']]
tableau_regions['écart LOW-HIGH'] = (tableau_regions['LOW'] - tableau_regions['HIGH']).round(2)

print('Taux de chômage moyen par région et niveau :')
display(tableau_regions.round(2))

print()
region_max_ecart = tableau_regions['écart LOW-HIGH'].idxmax()
region_min_ecart = tableau_regions['écart LOW-HIGH'].idxmin()
print(f'Région avec l\'écart le plus fort : {region_max_ecart} ({tableau_regions.loc[region_max_ecart, "écart LOW-HIGH"]:.1f} pts)')
print(f'Région avec l\'écart le plus faible : {region_min_ecart} ({tableau_regions.loc[region_min_ecart, "écart LOW-HIGH"]:.1f} pts)')
print()
print('→ RÉPONSE SQ1 : Non, le diplôme ne protège pas avec la même intensité selon la région.')

Taux de chômage moyen par région et niveau :


education_level,LOW,MEDIUM,HIGH,écart LOW-HIGH
region,,,,
Afrique,12.48,15.64,13.26,-0.78
Amériques,7.03,8.27,5.09,1.94
Asie-Pacifique,3.59,5.63,5.36,-1.77
Europe,14.28,8.91,5.47,8.81
Moyen-Orient,12.72,13.04,14.34,-1.62



Région avec l'écart le plus fort : Europe (8.8 pts)
Région avec l'écart le plus faible : Asie-Pacifique (-1.8 pts)

→ RÉPONSE SQ1 : Non, le diplôme ne protège pas avec la même intensité selon la région.


## 6. Zoom sur les diplômes universitaires (SQ2)

On repart des fichiers bruts pour accéder aux codes ISCED 5 à 8.

In [10]:
# Rechargement des fichiers bruts pour les codes ISCED détaillés
emp_raw = pd.read_csv(RAW_DIR / 'EMP_TEMP_SEX_EDU_NB_A.csv')
une_raw = pd.read_csv(RAW_DIR / 'UNE_TUNE_SEX_EDU_NB_A.csv')

# Codes du supérieur : ISCED 5 = Bac+2, 6 = Licence, 7 = Master, 8 = Doctorat
ISCED_SUP = ['EDU_ISCED11_5', 'EDU_ISCED11_6', 'EDU_ISCED11_7', 'EDU_ISCED11_8']
ISCED_LABELS = {
    'EDU_ISCED11_5': 'Bac+2 (ISCED 5)',
    'EDU_ISCED11_6': 'Licence (ISCED 6)',
    'EDU_ISCED11_7': 'Master (ISCED 7)',
    'EDU_ISCED11_8': 'Doctorat (ISCED 8)',
}

def filtrer_isced_sup(df_raw, col_valeur):
    """Extrait les lignes correspondant aux niveaux universitaires ISCED 5-8."""
    return df_raw[
        (df_raw['sex']       == 'SEX_T') &
        (df_raw['classif1'].isin(ISCED_SUP)) &
        (df_raw['obs_value'].notna()) &
        (df_raw['obs_status'] != 'U') &
        (df_raw['time'] >= 2000) & (df_raw['time'] <= 2023)
    ][['ref_area','time','classif1','obs_value']].rename(columns={
        'ref_area':'country','time':'year','classif1':'education_code','obs_value': col_valeur
    })

emp_sup = filtrer_isced_sup(emp_raw, 'nb_employes')
une_sup = filtrer_isced_sup(une_raw, 'nb_chomeurs')

# Fusion
df_sup = pd.merge(emp_sup, une_sup, on=['country','year','education_code'], how='inner')
df_sup['taux_chomage'] = df_sup['nb_chomeurs'] / (df_sup['nb_employes'] + df_sup['nb_chomeurs']) * 100
df_sup['education_label'] = df_sup['education_code'].map(ISCED_LABELS)

# Taux moyen par niveau universitaire
taux_sup = (
    df_sup.groupby('education_code')['taux_chomage']
    .mean()
    .reindex(ISCED_SUP)
    .round(2)
)

print('Taux de chômage moyen par niveau universitaire :')
for code, taux in taux_sup.items():
    print(f'  {ISCED_LABELS[code]:22s}: {taux:.2f}%')

print()
print(f'Écart Bac+2→Doctorat : {taux_sup["EDU_ISCED11_5"] - taux_sup["EDU_ISCED11_8"]:.2f} points')
print()
print('→ RÉPONSE SQ2 : Non, les diplômes universitaires ne sont pas équivalents.')
print('  Le Doctorat offre la meilleure protection contre le chômage.')

Taux de chômage moyen par niveau universitaire :
  Bac+2 (ISCED 5)       : 7.23%
  Licence (ISCED 6)     : 8.44%
  Master (ISCED 7)      : 5.86%
  Doctorat (ISCED 8)    : 3.07%

Écart Bac+2→Doctorat : 4.16 points

→ RÉPONSE SQ2 : Non, les diplômes universitaires ne sont pas équivalents.
  Le Doctorat offre la meilleure protection contre le chômage.


## 7. Synthèse numérique des résultats

In [11]:
print('=' * 65)
print('SYNTHÈSE CHIFFRÉE')
print('=' * 65)
print()
print('Taux de chômage moyen mondial (2000–2023) :')
for niveau, taux in taux_par_niveau.items():
    print(f'  {niveau:8s}: {taux:.1f}%')
print(f'  → Écart LOW–HIGH : {ecart:.1f} pts')
print()
print('Région avec l\'écart le plus marqué :', region_max_ecart)
print('Région avec l\'écart le plus faible :', region_min_ecart)
print()
print('Taux chômage par niveau universitaire :')
for code, taux in taux_sup.items():
    print(f'  {ISCED_LABELS[code]:22s}: {taux:.1f}%')
print('=' * 65)

SYNTHÈSE CHIFFRÉE

Taux de chômage moyen mondial (2000–2023) :
  LOW     : 10.9%
  MEDIUM  : 9.4%
  HIGH    : 6.9%
  → Écart LOW–HIGH : 4.0 pts

Région avec l'écart le plus marqué : Europe
Région avec l'écart le plus faible : Asie-Pacifique

Taux chômage par niveau universitaire :
  Bac+2 (ISCED 5)       : 7.2%
  Licence (ISCED 6)     : 8.4%
  Master (ISCED 7)      : 5.9%
  Doctorat (ISCED 8)    : 3.1%
